# Test 3: Out-of-Sample Prediction of the Post-GFC Structural Break**Paper:** Loss Aversion, Endogenous Reference Points, and Boom-Bust Asymmetry in Financial Wealth Dynamics  **Author:** Anurag Srivastava (Riskcare Ltd., London)  **Notebook purpose:** Out-of-sample validation of the model's structural predictions using the 2008 GFC regulatory break as a natural experiment.  **Data:** US Security Brokers and Dealers balance sheets — Federal Reserve Z.1 Financial Accounts, Table L.130.---## MotivationTests 1 and 2 estimate the model's structural parameters on the same data used to confirm them — they are in-sample. This test asks: **can the model, estimated entirely on pre-GFC data (1963–2007), predict post-GFC dealer balance sheet dynamics (2010–2025) out of sample?**The 2008 GFC and subsequent Basel III regulatory regime provide an exogenous structural break that the model was not designed to explain. The regulatory floor extension (equation 14 in the paper) adds a single parameter $\lambda^{\rm reg}$ that selectively binds in the expansion regime. The model makes three out-of-sample predictions that require **zero knowledge** of $\lambda^{\rm reg}$:| Prediction | What it says | Free parameters | Strongest test? ||---|---|---|---|| **A** | Post-GFC **contraction variance** = pre-GFC contraction variance (regulation doesn't affect contraction) | 0 | ✓ Quantitative || **B** | Post-GFC **persistence** $\hat{\beta}$ predicted by $1/(1+g_{\rm stagnation})^2$ using pre-GFC structural relationship | 0 | ✓ Quantitative || **C** | Variance ratio flips below 1 iff $\lambda^{\rm reg}$ exceeds a critical threshold computable from pre-GFC $\hat{\lambda}$ | 0 | Qualitative |If any of these predictions fail, the model's out-of-sample validity is falsified.

## 0. Setup

In [ ]:
# Install and import!pip install fredapi pandas numpy statsmodels scipy matplotlib --quietimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport statsmodels.api as smfrom statsmodels.tsa.stattools import adfullerfrom statsmodels.tsa.filters.hp_filter import hpfilterfrom scipy.stats import levene, f as f_distimport warningswarnings.filterwarnings('ignore')np.random.seed(2026)print("All packages loaded.")

## 1. FRED API KeyPaste your free FRED API key below.  Get one here (30 seconds): https://fred.stlouisfed.org/docs/api/api_key.html

In [ ]:
# ── Paste your FRED API key here ──────────────────────────────────────────────FRED_API_KEY = "YOUR_FRED_API_KEY_HERE"   # <── replace this stringfrom fredapi import Fredfred = Fred(api_key=FRED_API_KEY)print("FRED client initialised.")

## 2. Data Download and ConstructionSame Z.1 L.130 series as Tests 1 and 2. We construct the deviation state variable $D_t$ using both HP-filter and deterministic detrending.

In [ ]:
# ── Download Z.1 L.130 broker-dealer series ──────────────────────────────────print("Downloading Z.1 L.130 series from FRED...")series = {    "assets":  "BOGZ1FL664090005Q",   # Total financial assets    "equity":  "BOGZ1FL665080003Q",   # Proprietors' equity / net worth}raw = {}for name, sid in series.items():    try:        s = fred.get_series(sid)        s.name = name        raw[name] = s        print(f"  {sid} ({name}): {len(s)} obs, {s.index[0].date()} – {s.index[-1].date()}")    except Exception as e:        print(f"  ERROR fetching {sid}: {e}")df_raw = pd.DataFrame(raw).dropna()df_raw.index = pd.to_datetime(df_raw.index)df_raw = df_raw[(df_raw['assets'] > 0) & (df_raw['equity'] > 0)].copy()# Leveragedf_raw['leverage'] = df_raw['assets'] / df_raw['equity']df_raw = df_raw[(df_raw['leverage'] > 1) & (df_raw['leverage'] < 100)].copy()print(f"\nClean sample: {len(df_raw)} quarters, {df_raw.index[0].date()} – {df_raw.index[-1].date()}")print(f"Leverage range: {df_raw['leverage'].min():.1f} – {df_raw['leverage'].max():.1f}")

## 3. Define the Pre-GFC and Post-GFC SamplesThe split is at 2008Q1. The GFC window (2008Q1–2009Q4) is excluded from both samples to avoid contamination from acute crisis dynamics.- **Estimation sample (pre-GFC):** start – 2007Q4- **Holdout sample (post-GFC):** 2010Q1 – endThe model is estimated **only** on the pre-GFC sample. All post-GFC comparisons are out-of-sample predictions.

In [ ]:
# ── Sample split ──────────────────────────────────────────────────────────────PRE_GFC_END    = '2007-12-31'POST_GFC_START = '2010-01-01'df_pre  = df_raw[df_raw.index <= PRE_GFC_END].copy()df_post = df_raw[df_raw.index >= POST_GFC_START].copy()print(f"Pre-GFC  sample: {len(df_pre):>4} quarters, {df_pre.index[0].date()} – {df_pre.index[-1].date()}")print(f"Post-GFC sample: {len(df_post):>4} quarters, {df_post.index[0].date()} – {df_post.index[-1].date()}")print(f"GFC exclusion:   {len(df_raw) - len(df_pre) - len(df_post):>4} quarters")

## 4. Estimation on Pre-GFC Sample OnlyWe estimate the full model on the pre-GFC sample using both HP-filter and deterministic detrending. This produces:- $\hat{\lambda}_{\rm pre}$: implied loss aversion from the pre-GFC variance ratio- $\hat{V}^-_{\rm pre}$: contraction innovation variance (used for Prediction A)- $\hat{\phi}_{\rm pre}$: AR(1) persistence (used for Prediction B calibration)- $\hat{\beta}_{\rm pre} = \hat{\phi}^2_{\rm pre}$: persistence coefficient

In [ ]:
def estimate_sample(df_in, label, g_lit_annual=None):    '''Full regime AR(1) estimation on a given sample.        Returns dict with all structural estimates.    Uses HP-filter detrending by default; deterministic detrending if g_lit provided.    '''    df = df_in.copy()    res = {'label': label, 'n': len(df)}        if g_lit_annual is not None:        # Deterministic detrending on log(equity)        g_q = g_lit_annual / 4        log_eq = np.log(df['equity'].values)        t_idx = np.arange(len(df))        trend = g_q * t_idx        intercept = np.mean(log_eq - trend)        df['deviation'] = log_eq - trend - intercept        res['detrending'] = f'deterministic (g={g_lit_annual:.3f})'    else:        # HP filter detrending on log(leverage)        leverage_log = np.log(df['leverage'])        cycle, trend_log = hpfilter(leverage_log, lamb=1600)        df['deviation'] = cycle        res['detrending'] = 'HP filter (λ=1600)'        df['D_lag'] = df['deviation'].shift(1)    df['regime'] = (df['D_lag'] >= 0).astype(int)    df = df.dropna(subset=['D_lag'])        # Regime-specific AR(1)    for rname, rval in [('expansion', 1), ('contraction', 0)]:        mask = df['regime'] == rval        y = df.loc[mask, 'deviation'].values        y_lag = df.loc[mask, 'D_lag'].values        valid = ~(np.isnan(y) | np.isnan(y_lag))        y, y_lag = y[valid], y_lag[valid]                X = sm.add_constant(y_lag)        mod = sm.OLS(y, X).fit(cov_type='HC3')                resid = mod.resid        innov_var = np.var(resid, ddof=2)                res[f'{rname}_n'] = len(y)        res[f'{rname}_phi'] = mod.params[1]        res[f'{rname}_phi_se'] = mod.bse[1]        res[f'{rname}_var'] = innov_var        res[f'{rname}_std'] = np.sqrt(innov_var)        res[f'{rname}_resid'] = resid        # Pooled AR(1)    y_all = df['deviation'].values    y_lag_all = df['D_lag'].values    valid = ~(np.isnan(y_all) | np.isnan(y_lag_all))    X_pool = sm.add_constant(y_lag_all[valid])    mod_pool = sm.OLS(y_all[valid], X_pool).fit(cov_type='HC3')    res['phi_pooled'] = mod_pool.params[1]    res['phi_pooled_se'] = mod_pool.bse[1]        # Structural parameters    res['R_hat'] = res['expansion_var'] / res['contraction_var']    res['lambda_hat'] = res['R_hat'] ** 0.25    res['asym_ratio'] = (res['R_hat'] - 1) / res['R_hat']    res['beta_hat'] = res['phi_pooled'] ** 2        return res# ── Estimate on pre-GFC sample ───────────────────────────────────────────────pre_hp = estimate_sample(df_pre, 'Pre-GFC (HP filter)')# Deterministic detrending with pre-GFC average gG_PRE_GFC = 0.012   # 1.2% p.a. average TFP growth 1952-2007 (Fernald 2015)pre_det = estimate_sample(df_pre, 'Pre-GFC (deterministic)', g_lit_annual=G_PRE_GFC)print("=" * 70)print("PRE-GFC ESTIMATION (ESTIMATION SAMPLE ONLY)")print("=" * 70)for est in [pre_hp, pre_det]:    print(f"\n  {est['label']}  (N={est['n']})")    print(f"  Detrending: {est['detrending']}")    print(f"  Expansion:   V⁺ = {est['expansion_var']:.6f}  (N={est['expansion_n']})")    print(f"  Contraction: V⁻ = {est['contraction_var']:.6f}  (N={est['contraction_n']})")    print(f"  Variance ratio R̂ = {est['R_hat']:.4f}")    print(f"  Implied λ̂ = {est['lambda_hat']:.4f}")    print(f"  |γ|/α = {est['asym_ratio']:.4f}")    print(f"  Pooled φ̂ = {est['phi_pooled']:.4f} (se={est['phi_pooled_se']:.4f})")    print(f"  β̂ = φ̂² = {est['beta_hat']:.4f}")

## 5. Post-GFC Observed Values (Holdout Sample)Now we estimate the same statistics on the post-GFC holdout sample. These are the **observed** values that the pre-GFC model must predict. We do not use any pre-GFC parameter estimates in this estimation — it is purely descriptive.

In [ ]:
# ── Estimate on post-GFC sample ───────────────────────────────────────────────post_hp = estimate_sample(df_post, 'Post-GFC (HP filter)')G_POST_GFC = 0.003   # 0.3% p.a. secular stagnation (Fernald 2015 / CBO)post_det = estimate_sample(df_post, 'Post-GFC (deterministic)', g_lit_annual=G_POST_GFC)print("=" * 70)print("POST-GFC OBSERVED VALUES (HOLDOUT SAMPLE)")print("=" * 70)for est in [post_hp, post_det]:    print(f"\n  {est['label']}  (N={est['n']})")    print(f"  Detrending: {est['detrending']}")    print(f"  Expansion:   V⁺ = {est['expansion_var']:.6f}  (N={est['expansion_n']})")    print(f"  Contraction: V⁻ = {est['contraction_var']:.6f}  (N={est['contraction_n']})")    print(f"  Variance ratio R̂ = {est['R_hat']:.4f}")    print(f"  Pooled φ̂ = {est['phi_pooled']:.4f}")    print(f"  β̂ = φ̂² = {est['beta_hat']:.4f}")

## 6. Prediction A: Contraction Variance Invariance**Model prediction:** The regulatory floor $\lambda^{\rm reg}$ only binds in the expansion regime (equation 14). In contraction, $\lambda^{\rm eff}_p = \lambda\lambda^0_p$ — the same as pre-GFC. Therefore:$$\hat{V}^-_{\rm post} = \hat{V}^-_{\rm pre}$$This is the strongest out-of-sample test: zero free parameters, quantitative, and directly falsifiable.**Test:** We compare $\hat{V}^-_{\rm pre}$ (prediction) to $\hat{V}^-_{\rm post}$ (observed) and report the prediction error as a percentage. We also test equality of the pre- and post-GFC contraction residual variances using the Levene test.

In [ ]:
print("=" * 70)print("PREDICTION A: CONTRACTION VARIANCE INVARIANCE")print("=" * 70)print()print("Model prediction: regulation does not affect contraction dynamics.")print("Therefore V⁻(post-GFC) should equal V⁻(pre-GFC).")print()for det_label, pre, post in [('HP filter', pre_hp, post_hp),                               ('Deterministic', pre_det, post_det)]:    predicted = pre['contraction_var']    observed  = post['contraction_var']    pct_error = 100 * (observed - predicted) / predicted        # Levene test for equality of contraction variances    lev_stat, lev_pval = levene(pre['contraction_resid'], post['contraction_resid'])        # F-test (two-sided)    n1 = len(pre['contraction_resid'])    n2 = len(post['contraction_resid'])    F_stat = observed / predicted    F_pval = 2 * min(f_dist.cdf(F_stat, n2-2, n1-2), 1 - f_dist.cdf(F_stat, n2-2, n1-2))        print(f"  [{det_label}]")    print(f"    Predicted V⁻ (from pre-GFC):  {predicted:.6f}")    print(f"    Observed  V⁻ (post-GFC):      {observed:.6f}")    print(f"    Prediction error:              {pct_error:+.1f}%")    print(f"    F-test (H₀: V⁻_pre = V⁻_post): F = {F_stat:.3f}, p = {F_pval:.4f}")    print(f"    Levene test:                    W = {lev_stat:.3f}, p = {lev_pval:.4f}")        if abs(pct_error) < 50 and lev_pval > 0.05:        print(f"    ✓ PASS: Cannot reject equality — contraction variance is invariant to regulation")    elif lev_pval > 0.05:        print(f"    ~ MARGINAL: Cannot reject equality, but prediction error is {abs(pct_error):.0f}%")    else:        print(f"    ✗ FAIL: Contraction variance differs significantly pre vs post GFC")    print()

## 7. Prediction B: Persistence Depends Only on Growth Rate**Model prediction:** $\beta \approx 1/(1+g)^2$, independent of $\lambda$ and $\lambda^{\rm reg}$ (separation property). Using the pre-GFC estimated structural relationship and the post-GFC growth rate $g_{\rm stagnation} = 0.3\%$:$$\hat{\beta}_{\rm predicted} = \frac{1}{(1 + 0.003)^2} = 0.9940$$This prediction uses only the model's functional form and an externally calibrated growth rate — no pre-GFC parameter estimation is needed (the prediction is purely structural).

In [ ]:
print("=" * 70)print("PREDICTION B: PERSISTENCE DEPENDS ONLY ON GROWTH RATE")print("=" * 70)print()print("Model prediction: β = 1/(1+g)², independent of λ and regulation.")print(f"Post-GFC g_lit = {G_POST_GFC:.3f} → predicted β = {1/(1+G_POST_GFC)**2:.6f}")print()beta_predicted = 1 / (1 + G_POST_GFC) ** 2for det_label, post in [('HP filter', post_hp), ('Deterministic', post_det)]:    beta_observed = post['beta_hat']    phi_observed = post['phi_pooled']    phi_predicted = 1 / (1 + G_POST_GFC)   # quarterly        pct_error_beta = 100 * (beta_observed - beta_predicted) / beta_predicted    pct_error_phi = 100 * (phi_observed - phi_predicted) / phi_predicted        print(f"  [{det_label}]")    print(f"    Predicted φ = 1/(1+g):         {phi_predicted:.6f}")    print(f"    Observed  φ̂ (post-GFC):        {phi_observed:.6f}")    print(f"    φ prediction error:            {pct_error_phi:+.2f}%")    print(f"    Predicted β = 1/(1+g)²:        {beta_predicted:.6f}")    print(f"    Observed  β̂ = φ̂² (post-GFC):  {beta_observed:.6f}")    print(f"    β prediction error:            {pct_error_beta:+.2f}%")        # Note on HP filter    if 'HP' in det_label:        print(f"    ⚠ HP filter absorbs low-frequency persistence — deterministic detrending is")        print(f"      the appropriate test (see Finding 2 in the paper)")    else:        if abs(pct_error_phi) < 5:            print(f"    ✓ PASS: Persistence within {abs(pct_error_phi):.1f}% of structural prediction")        else:            print(f"    ~ MARGINAL: Persistence {abs(pct_error_phi):.1f}% from structural prediction")    print()print("Note: The HP-filter result is expected to show poor persistence recovery")print("(see Finding 2 discussion). The deterministic detrending result is the")print("appropriate out-of-sample test of Prediction B.")

## 8. Prediction C: Variance Ratio Sign Flip**Model prediction:** The variance ratio under regulation is:$$\mathcal{R}^{\rm reg} = \frac{\lambda^2 \lambda_p^{0\,4}}{\lambda^{\rm reg\,2}}$$This falls below 1 when $\lambda^{\rm reg} > \lambda \lambda_p^{0\,2}$. Using the pre-GFC $\hat{\lambda}_{\rm pre}$, we can compute the critical regulatory threshold without any post-GFC information.The test is qualitative: does $\hat{\mathcal{R}}_{\rm post} < 1$, consistent with binding regulation?

In [ ]:
print("=" * 70)print("PREDICTION C: VARIANCE RATIO SIGN FLIP UNDER REGULATION")print("=" * 70)print()for det_label, pre, post in [('HP filter', pre_hp, post_hp),                               ('Deterministic', pre_det, post_det)]:    lam_pre = pre['lambda_hat']    R_post  = post['R_hat']        # The model predicts R flips below 1 when regulation binds in expansion    # Pre-GFC: R = λ⁴ > 1 (no regulation)    # Post-GFC: R < 1 iff λ^reg > λ_p^0 / λ (regulation compresses expansion variance)        print(f"  [{det_label}]")    print(f"    Pre-GFC implied λ̂:              {lam_pre:.4f}")    print(f"    Pre-GFC R̂ = λ̂⁴:                {pre['R_hat']:.4f}")    print(f"    Post-GFC observed R̂:            {R_post:.4f}")    print()        if R_post < 1:        # Back out the implied λ^reg / λ_p^0 ratio from post-GFC R        # R_reg = (λ_p^0 / λ_reg)² · λ² → λ_reg/λ_p^0 = λ / sqrt(R_reg)        implied_reg_ratio = lam_pre / np.sqrt(R_post)        print(f"    ✓ PASS: R̂_post < 1 — consistent with binding regulation")        print(f"    Implied λ^reg/λ_p^0 = {implied_reg_ratio:.3f} > 1 (regulation dominates)")        print(f"    The pre-GFC λ̂ = {lam_pre:.4f} predicted that any regulation")        print(f"    with λ^reg > λ_p^0/{lam_pre:.4f} = λ_p^0 × {1/lam_pre:.4f}")        print(f"    would flip the variance ratio below 1.")    else:        print(f"    ✗ FAIL: R̂_post ≥ 1 — no evidence of binding regulation")    print()

## 9. Summary: Out-of-Sample ScorecardAll three predictions are evaluated together. The model is estimated entirely on pre-GFC data (1963–2007); predictions are compared to post-GFC observations (2010–2025).

In [ ]:
print("=" * 70)print("OUT-OF-SAMPLE SCORECARD")print("Model estimated on pre-GFC data only (1963-2007)")print("Predictions compared to post-GFC holdout (2010-2025)")print("=" * 70)# Use deterministic detrending as the primary specificationpre  = pre_detpost = post_det# Prediction AV_neg_pred = pre['contraction_var']V_neg_obs  = post['contraction_var']err_A = 100 * (V_neg_obs - V_neg_pred) / V_neg_predlev_A_stat, lev_A_pval = levene(pre['contraction_resid'], post['contraction_resid'])# Prediction Bbeta_pred_B = 1 / (1 + G_POST_GFC) ** 2beta_obs_B  = post['beta_hat']err_B = 100 * (beta_obs_B - beta_pred_B) / beta_pred_Bphi_pred_B = 1 / (1 + G_POST_GFC)phi_obs_B  = post['phi_pooled']err_B_phi = 100 * (phi_obs_B - phi_pred_B) / phi_pred_B# Prediction CR_post = post['R_hat']flip_C = R_post < 1print()print(f"{'Prediction':<12} {'Description':<42} {'Predicted':>10} {'Observed':>10} {'Error':>8} {'Result':>8}")print("-" * 92)# Apass_A = lev_A_pval > 0.05print(f"{'A':<12} {'Contraction variance V⁻':<42} {V_neg_pred:>10.6f} {V_neg_obs:>10.6f} {err_A:>+7.1f}% {'PASS' if pass_A else 'FAIL':>8}")# B (phi)pass_B = abs(err_B_phi) < 5print(f"{'B (φ)':<12} {'Persistence φ = 1/(1+g)':<42} {phi_pred_B:>10.6f} {phi_obs_B:>10.6f} {err_B_phi:>+7.2f}% {'PASS' if pass_B else 'FAIL':>8}")# B (beta)print(f"{'B (β)':<12} {'Persistence β = 1/(1+g)²':<42} {beta_pred_B:>10.6f} {beta_obs_B:>10.6f} {err_B:>+7.2f}% {'':>8}")# Cprint(f"{'C':<12} {'Variance ratio flips below 1':<42} {'R < 1':>10} {R_post:>10.4f} {'':>8} {'PASS' if flip_C else 'FAIL':>8}")print("-" * 92)n_pass = sum([pass_A, pass_B, flip_C])print(f"\nResult: {n_pass}/3 predictions confirmed out of sample")print(f"\nNote: All predictions use zero post-GFC parameters.")print(f"Pre-GFC λ̂ = {pre['lambda_hat']:.4f}, g_stagnation = {G_POST_GFC}")

## 10. Bootstrap Confidence Intervals for Prediction ATo properly quantify uncertainty in the contraction variance comparison, we bootstrap both samples and compute the distribution of the prediction error $\hat{V}^-_{\rm post} - \hat{V}^-_{\rm pre}$.

In [ ]:
# ── Bootstrap for Prediction A ────────────────────────────────────────────────N_BOOT = 5000def bootstrap_contraction_var(resid, n_boot=N_BOOT):    '''Bootstrap the contraction innovation variance.'''    n = len(resid)    boot_vars = np.zeros(n_boot)    for b in range(n_boot):        idx = np.random.choice(n, size=n, replace=True)        boot_vars[b] = np.var(resid[idx], ddof=2)    return boot_varsboot_pre  = bootstrap_contraction_var(pre_det['contraction_resid'])boot_post = bootstrap_contraction_var(post_det['contraction_resid'])# Distribution of prediction errorboot_diff = boot_post - boot_preboot_pct  = 100 * boot_diff / boot_preci_diff = np.percentile(boot_diff, [2.5, 97.5])ci_pct  = np.percentile(boot_pct, [2.5, 97.5])print("=" * 70)print("BOOTSTRAP CONFIDENCE INTERVALS — PREDICTION A")print(f"(Deterministic detrending, {N_BOOT} bootstrap replications)")print("=" * 70)print()print(f"  V⁻_post - V⁻_pre:")print(f"    Point estimate: {V_neg_obs - V_neg_pred:.6f}")print(f"    95% CI: [{ci_diff[0]:.6f}, {ci_diff[1]:.6f}]")print(f"    Contains zero: {'Yes ✓' if ci_diff[0] <= 0 <= ci_diff[1] else 'No ✗'}")print()print(f"  Percentage prediction error:")print(f"    Point estimate: {err_A:+.1f}%")print(f"    95% CI: [{ci_pct[0]:+.1f}%, {ci_pct[1]:+.1f}%]")

## 11. Figure: Out-of-Sample Prediction Summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))fig.suptitle(    "Test 3: Out-of-Sample Prediction of Post-GFC Structural Break\n"    "Model estimated on pre-GFC data (1963–2007); predictions vs post-GFC holdout (2010–2025)",    fontsize=11, fontweight='bold', y=1.02)col_pre  = '#2980B9'col_post = '#C0392B'col_pred = '#27AE60'# ── Panel A: Contraction variance ────────────────────────────────────────────ax = axes[0]bars = ax.bar(['Pre-GFC\n(estimated)', 'Post-GFC\n(predicted)', 'Post-GFC\n(observed)'],              [V_neg_pred, V_neg_pred, V_neg_obs],              color=[col_pre, col_pred, col_post],              alpha=[0.8, 0.4, 0.8],              edgecolor=['none', col_pred, 'none'],              linewidth=[0, 2, 0],              linestyle=['solid', 'dashed', 'solid'])ax.set_ylabel('Contraction innovation variance V⁻')ax.set_title('Prediction A:\nContraction variance', fontsize=10, fontweight='bold')ax.text(0.5, 0.95, f'Error: {err_A:+.1f}%', transform=ax.transAxes,        ha='center', va='top', fontsize=9,        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.8))# ── Panel B: Persistence ────────────────────────────────────────────────────ax = axes[1]bars = ax.bar(['Predicted\nβ=1/(1+g)²', 'Observed\nβ̂ (post-GFC)'],              [beta_pred_B, beta_obs_B],              color=[col_pred, col_post], alpha=0.8)ax.set_ylabel('Persistence coefficient β')ax.set_title('Prediction B:\nPersistence', fontsize=10, fontweight='bold')ax.text(0.5, 0.95, f'Error: {err_B:+.1f}%', transform=ax.transAxes,        ha='center', va='top', fontsize=9,        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.8))ax.axhline(1.0, color='gray', ls=':', lw=0.8, label='Unit root')# ── Panel C: Variance ratio sign flip ───────────────────────────────────────ax = axes[2]bars = ax.bar(['Pre-GFC\nR̂', 'Post-GFC\nR̂'],              [pre['R_hat'], R_post],              color=[col_pre, col_post], alpha=0.8)ax.axhline(1.0, color='black', ls='--', lw=1.0, label='R = 1 (symmetric)')ax.set_ylabel('Variance ratio R̂ = V⁺/V⁻')ax.set_title('Prediction C:\nVariance ratio flip', fontsize=10, fontweight='bold')ax.legend(fontsize=8)plt.tight_layout()plt.savefig('fig5_oos_prediction.pdf', dpi=300, bbox_inches='tight')plt.show()print("Figure saved as fig5_oos_prediction.pdf")

## 12. Structured Results OutputJSON block for cross-referencing with the manuscript.

In [ ]:
import json as _json_oos_out = {    "test": "test3_oos_prediction",    "estimation_sample": {        "start": str(df_pre.index[0].date()),        "end": str(df_pre.index[-1].date()),        "n_quarters": len(df_pre),        "detrending": "deterministic",        "g_lit_annual": G_PRE_GFC    },    "holdout_sample": {        "start": str(df_post.index[0].date()),        "end": str(df_post.index[-1].date()),        "n_quarters": len(df_post),        "g_lit_annual": G_POST_GFC    },    "pre_gfc_estimates": {        "lambda_hat": round(pre_det['lambda_hat'], 4),        "R_hat": round(pre_det['R_hat'], 4),        "contraction_var": round(pre_det['contraction_var'], 6),        "phi_pooled": round(pre_det['phi_pooled'], 6),        "beta_hat": round(pre_det['beta_hat'], 6)    },    "prediction_A": {        "description": "Contraction variance invariant to regulation",        "predicted": round(float(V_neg_pred), 6),        "observed": round(float(V_neg_obs), 6),        "pct_error": round(float(err_A), 1),        "levene_pval": round(float(lev_A_pval), 4),        "bootstrap_ci_pct": [round(float(ci_pct[0]), 1), round(float(ci_pct[1]), 1)],        "pass": bool(lev_A_pval > 0.05)    },    "prediction_B": {        "description": "Persistence β = 1/(1+g)²",        "predicted_beta": round(float(beta_pred_B), 6),        "observed_beta": round(float(beta_obs_B), 6),        "predicted_phi": round(float(phi_pred_B), 6),        "observed_phi": round(float(phi_obs_B), 6),        "pct_error_phi": round(float(err_B_phi), 2),        "pass": bool(abs(err_B_phi) < 5)    },    "prediction_C": {        "description": "Variance ratio flips below 1 under regulation",        "pre_gfc_R": round(float(pre_det['R_hat']), 4),        "post_gfc_R": round(float(R_post), 4),        "flipped": bool(R_post < 1),        "pass": bool(R_post < 1)    }}print(_json.dumps(_oos_out, indent=2))

## 13. Data Sources and Citations**Primary data:**Board of Governors of the Federal Reserve System (US),  *Security Brokers and Dealers; Total Financial Assets, Level* [BOGZ1FL664090005Q],  retrieved from FRED, Federal Reserve Bank of St. Louis;  https://fred.stlouisfed.org/series/BOGZ1FL664090005QBoard of Governors of the Federal Reserve System (US),  *Security Brokers and Dealers; Proprietors' Equity with IVA, Level* [BOGZ1FL665080003Q],  retrieved from FRED, Federal Reserve Bank of St. Louis;  https://fred.stlouisfed.org/series/BOGZ1FL665080003QRelease: Z.1 Financial Accounts of the United States  https://www.federalreserve.gov/releases/z1/**Growth rate calibration:**Fernald, J. (2015). Productivity and Potential Output Before, During, and After the Great Recession.  *NBER Macroeconomics Annual*, 29(1), 1–51.Congressional Budget Office (2013). *The Budget and Economic Outlook: Fiscal Years 2013 to 2023.*  Washington, DC: CBO.